# Ferreus RMT to geoh5 proof of concept

Objectives:

Explore if the packages allows for

    exclusion constraints

    orientation constraints

POC with notebook creating a surface transferred to geoh5

In [ ]:
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from ferreus_rbf import RBFInterpolator
from ferreus_rbf.interpolant_config import (
    Drift,
    InterpolantSettings,
    RBFKernelType,
)
from ferreus_rmt import (
    BoundaryClosure,
    ClusterMethod,
    build_isosurface,
)
from geoh5py.groups import ContainerGroup
from geoh5py.objects import Points, Surface
from geoh5py.workspace import Workspace

### define a known sphere

In [ ]:
CENTER = np.array([0.0, 0.0, 0.0], dtype=np.float64)
RADIUS = 50.0


def sphere_values(points: np.ndarray) -> np.ndarray:
    """negative inside the sphere, zero on it, positive outside."""

    offsets = points - CENTER
    return np.linalg.norm(offsets, axis=1) - RADIUS


def sphere_values_and_gradients(
    points: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """return values and analytic gradients."""
    offsets = points - CENTER
    distances = np.linalg.norm(offsets, axis=1)

    gradients = np.divide(
        offsets,
        distances[:, None],
        out=np.zeros_like(offsets),
        where=distances[:, None] > 0.0,
    )

    return distances - RADIUS, gradients

### extract the surface with rmt

In [ ]:
seed_points = np.array(
    [
        [50.0, 0.0, 0.0],
        [-50.0, 0.0, 0.0],
    ],
    dtype=np.float64,
)

# Define the axis-aligned bounding box extents to extract the isosurface within:
# [xmin, ymin, zmin, xmax, ymax, zmax].
extents = np.array(
    [-75.0, -75.0, -75.0, 75.0, 75.0, 75.0],
    dtype=np.float64,
)


resolution = 5.0
isovalue = 0.0

mesh = build_isosurface(
    seed_points,  # approx. locations of the surface to seed the wavefront
    extents,  # the axis-aligned bounding box limiting extraction
    resolution,  # lattice spacing in coordinate units - smaller means more detail
    isovalue,  # the scalar value defining the surface
    surface_fn=sphere_values,  # surface_fn (sphere_values) is the scalar function that defines the geometry
    gradient_fn=sphere_values_and_gradients,  #
)

vertices = np.asarray(mesh.vertices)
cells = np.asarray(mesh.facets, dtype=np.uint32)

print("Vertices:", vertices.shape)
print("Triangles:", cells.shape)

### validate the result

In [ ]:
radii = np.linalg.norm(vertices - CENTER, axis=1)
radius_errors = radii - RADIUS

print("Minimum radius:", radii.min())
print("Maximum radius:", radii.max())
print("Mean absolute radius error:", np.abs(radius_errors).mean())
print("Maximum absolute radius error:", np.abs(radius_errors).max())

assert cells.ndim == 2 and cells.shape[1] == 3
assert vertices.ndim == 2 and vertices.shape[1] == 3
assert np.abs(radius_errors).max() < resolution

# numbers should be close to the radius, but likely not exact because of the resolution of the lattice.

### show the sphere we've constructed

In [ ]:
figure = plt.figure(figsize=(8, 7))
axis = figure.add_subplot(111, projection="3d")

axis.plot_trisurf(
    vertices[:, 0],
    vertices[:, 1],
    vertices[:, 2],
    triangles=cells,
    linewidth=0.05,
    alpha=0.85,
)

axis.set_xlabel("X")
axis.set_ylabel("Y")
axis.set_zlabel("Z")
axis.set_box_aspect(np.ptp(vertices, axis=0))
axis.set_title("Analytic sphere extracted with Ferreus RMT")

plt.show()

### Export to geoh5

In [ ]:
output_directory = Path.cwd() / "ferreus_output"
output_directory.mkdir(exist_ok=True)

output_path = output_directory / (
    f"ferreus_rmt_poc_{datetime.now():%Y%m%d_%H%M%S}.geoh5"
)

with Workspace.create(output_path) as workspace:
    group = ContainerGroup.create(
        workspace,
        name="Ferreus RMT POC",
    )

    Surface.create(
        workspace,
        name="RMT analytic sphere",
        vertices=vertices,
        cells=cells,
        parent=group,
    )

print("Created:", output_path)

## Fit a synthetic folded horizon with ferreus_rbf

The analytic sphere proved that `ferreus_rmt` can extract a known scalar
field and that its mesh can be exported to geoH5.

This section replaces the analytic sphere field with an RBF field fitted
from scattered scalar observations.

A known analytic geological field is retained as ground truth, allowing
the interpolation and resulting surface errors to be measured.

In [ ]:
MODEL_EXTENTS = np.array(
    [-100.0, -100.0, -60.0, 100.0, 100.0, 60.0],
    dtype=np.float64,
)


def true_horizon_elevation(
    x: np.ndarray,
    y: np.ndarray,
) -> np.ndarray:
    """Elevation of the synthetic folded horizon."""
    return 20.0 * np.sin(x / 35.0) + 10.0 * np.cos(y / 40.0)


def true_geological_field(points: np.ndarray) -> np.ndarray:
    """Signed vertical separation from the synthetic horizon."""
    horizon_z = true_horizon_elevation(
        points[:, 0],
        points[:, 1],
    )

    return points[:, 2] - horizon_z

### Generate scattered scalar observations

In [ ]:
random_generator = np.random.default_rng(42)

number_of_source_points = 400

source_points = random_generator.uniform(
    low=MODEL_EXTENTS[:3],
    high=MODEL_EXTENTS[3:],
    size=(number_of_source_points, 3),
).astype(np.float64)

source_values = true_geological_field(source_points).astype(np.float64)

print("Source points:", source_points.shape)
print("Source values:", source_values.shape)
print(
    "Value range:",
    float(source_values.min()),
    "to",
    float(source_values.max()),
)

### Fit the RBF field

In [ ]:
rbf_settings = InterpolantSettings(
    kernel_type=RBFKernelType.Linear,  # selects the radial kernel used to interpolate between observations.
    drift=Drift.Linear,  # adds a broad linear trend, useful here because the field contains a strong linear z component.
    nugget=0.0,  # 0.0 requests an unsmoothed fit. Fine for synthetic observations as they contain no noise.
)

rbf_model = RBFInterpolator(
    points=source_points,
    values=source_values,
    interpolant_settings=rbf_settings,
)

### Check the training residuals

In [ ]:
source_predictions = np.asarray(rbf_model.evaluate_at_source(add_nugget=False)).reshape(
    -1
)

source_residuals = source_predictions - source_values

print(
    "Source RMSE:",
    float(np.sqrt(np.mean(source_residuals**2))),
)
print(
    "Maximum source error:",
    float(np.max(np.abs(source_residuals))),
)

# This demonstrates how closely the solved RBF reproduces the values used to fit it.
# For a zero-nugget, synthetic problem, the residuals should be very small, on the order of machine precision.

### Test at locations not used for fitting

In [ ]:
number_of_validation_points = 200

validation_points = random_generator.uniform(
    low=MODEL_EXTENTS[:3],
    high=MODEL_EXTENTS[3:],
    size=(number_of_validation_points, 3),
).astype(np.float64)

true_validation_values = true_geological_field(validation_points)

predicted_validation_values = np.asarray(rbf_model.evaluate(validation_points)).reshape(
    -1
)

validation_residuals = predicted_validation_values - true_validation_values

validation_rmse = np.sqrt(np.mean(validation_residuals**2))

print("Validation RMSE:", float(validation_rmse))
print(
    "Maximum validation error:",
    float(np.max(np.abs(validation_residuals))),
)

# The source residual measures fitting accuracy, i.e. an RMSE of 0.69 means approx 0.69 vertical units
# The validation residual measures interpolation accuracy between observations

### Prepare the RBF for repeated RMT evaluations

In [ ]:
folded_resolution = 5.0

# RMT evaluates a small halo of lattice positions around the extraction
# box. Give the cached RBF evaluator enough room for those calls.
evaluator_padding = 5.0 * folded_resolution

evaluator_extents = MODEL_EXTENTS + np.array(
    [
        -evaluator_padding,
        -evaluator_padding,
        -evaluator_padding,
        evaluator_padding,
        evaluator_padding,
        evaluator_padding,
    ],
    dtype=np.float64,
)

rbf_model.build_evaluator(evaluator_extents)

### Choose seed points for the folded horizon

RMT is a surface-following algorithm. This means it does not scan every cell in the model volume. Instead, it starts near the requested isosurface at the seed points and follows the connected surface away from them.

These seeds are algorithm inputs, not extra observations used to fit the RBF. In this synthetic example the true horizon is known, so a small grid of exact surface locations is convenient. In a real geological workflow, observed contact points with scalar value zero would be natural seeds.

Multiple seeds are useful when a model may contain disconnected surface components. A component that has no nearby seed may not be discovered.

In [ ]:
seed_coordinates = np.linspace(-80.0, 80.0, 5)
seed_x, seed_y = np.meshgrid(
    seed_coordinates,
    seed_coordinates,
    indexing="xy",
)

folded_seed_points = np.column_stack(
    [
        seed_x.ravel(),
        seed_y.ravel(),
        true_horizon_elevation(
            seed_x.ravel(),
            seed_y.ravel(),
        ),
    ]
).astype(np.float64)

print("RMT seed points shape:", folded_seed_points.shape)
print(
    "Maximum absolute seed field value:",
    float(np.max(np.abs(true_geological_field(folded_seed_points)))),
)

### Extract the fitted zero-isosurface with RMT

The requested surface is where the fitted scalar field equals folded_isovalue, here zero. Because the synthetic field is z - horizon_z, zero represents the folded horizon, negative values lie below it, and positive values lie above it.

folded_resolution is the spacing of RMT's internal sampling lattice in the same coordinate units as X, Y, and Z. 5.0 means samples are approximately five units apart. A smaller value can preserve finer geometry, but requires more field evaluations and usually produces more triangles.

evaluate_targets and evaluate_targets_with_gradients use the cached evaluator prepared above. The gradients help RMT project its working vertices toward the requested scalar value; they are not geological orientation observations.

In [ ]:
folded_isovalue = 0.0

folded_mesh = build_isosurface(
    seed_points=folded_seed_points,
    extents=MODEL_EXTENTS,
    resolution=folded_resolution,
    isovalue=folded_isovalue,
    surface_fn=rbf_model.evaluate_targets,
    gradient_fn=rbf_model.evaluate_targets_with_gradients,
    cluster_method=ClusterMethod.CurvatureWeighted,
    boundary_closure=BoundaryClosure.None_,
)

folded_vertices = np.asarray(
    folded_mesh.vertices,
    dtype=np.float64,
)
folded_cells = np.asarray(
    folded_mesh.facets,
    dtype=np.uint32,
)

print("Folded surface vertices:", folded_vertices.shape)
print("Folded surface triangles:", folded_cells.shape)

### Validate both parts of the workflow

There are two distinct errors worth checking:

1. **RMT extraction error:** evaluate the fitted RBF at every mesh vertex. These values should be close to the requested isovalue of zero. This checks whether RMT followed the field it was given.
2. **Modelling error:** evaluate the known analytic field at every mesh vertex. Here that value is also the signed vertical distance from the true horizon. This checks how closely the RBF-derived surface matches the ground truth between the observations.

Keeping these checks separate matters as a mesh can follow an imperfect RBF field very accurately.

In [ ]:
fitted_values_at_vertices = np.asarray(
    rbf_model.evaluate(folded_vertices),
    dtype=np.float64,
).reshape(-1)

rmt_isovalue_errors = fitted_values_at_vertices - folded_isovalue

# For this particular synthetic field, this is the signed vertical
# separation between a mesh vertex and the true folded horizon.
true_vertical_errors = true_geological_field(folded_vertices)
absolute_vertical_errors = np.abs(true_vertical_errors)

print(
    "Maximum absolute RMT isovalue error:",
    float(np.max(np.abs(rmt_isovalue_errors))),
)
print(
    "Mean absolute vertical surface error:",
    float(np.mean(absolute_vertical_errors)),
)
print(
    "Vertical error percentiles (50%, 90%, 95%, 100%):",
    np.percentile(absolute_vertical_errors, [50, 90, 95, 100]),
)

assert folded_vertices.ndim == 2 and folded_vertices.shape[1] == 3
assert folded_cells.ndim == 2 and folded_cells.shape[1] == 3
assert np.all(np.isfinite(folded_vertices))
assert np.all(np.isfinite(fitted_values_at_vertices))

### Inspect the fitted surface and nearby observations

Only observations reasonably close to the horizon are plotted, which keeps the diagnostic readable. Their colours show the signed scalar values supplied to the RBF: blue points are below the horizon, red points are above it, and values near zero lie close to the interpreted contact.

In [ ]:
near_horizon = np.abs(source_values) <= 15.0

figure = plt.figure(figsize=(10, 8))
axis = figure.add_subplot(111, projection="3d")

axis.plot_trisurf(
    folded_vertices[:, 0],
    folded_vertices[:, 1],
    folded_vertices[:, 2],
    triangles=folded_cells,
    color="lightsteelblue",
    linewidth=0.05,
    alpha=0.70,
)

observation_plot = axis.scatter(
    source_points[near_horizon, 0],
    source_points[near_horizon, 1],
    source_points[near_horizon, 2],
    c=source_values[near_horizon],
    cmap="coolwarm",
    vmin=-15.0,
    vmax=15.0,
    s=20,
    edgecolors="black",
    linewidths=0.2,
)

figure.colorbar(
    observation_plot,
    ax=axis,
    shrink=0.65,
    pad=0.10,
    label="Observed signed field value",
)

axis.set_xlabel("X")
axis.set_ylabel("Y")
axis.set_zlabel("Z")
axis.set_box_aspect(MODEL_EXTENTS[3:] - MODEL_EXTENTS[:3])
axis.view_init(elev=25, azim=-55)
axis.set_title("RBF folded horizon extracted with Ferreus RMT")

plt.show()

### Export the folded surface and its observations to geoh5

The output contains:

- a Points object holding every scalar observation used to fit the RBF;
- the interpolated Surface object produced by RMT;
- the fitted RBF value on each surface vertex, useful for checking extraction; and
- the signed vertical error relative to the known synthetic horizon, useful only because this POC has ground truth.

In Geoscience ANALYST, colour the observation points by **Implicit scalar value** and the surface by **True vertical error** to inspect the result.

In [ ]:
output_directory = Path.cwd() / "ferreus_output"
output_directory.mkdir(exist_ok=True)

folded_output_path = output_directory / (
    f"ferreus_rbf_folded_horizon_{datetime.now():%Y%m%d_%H%M%S}.geoh5"
)

with Workspace.create(folded_output_path) as workspace:
    group = ContainerGroup.create(
        workspace,
        name="Ferreus folded horizon POC",
    )

    observations = Points.create(
        workspace,
        name="RBF scalar observations",
        vertices=source_points,
        parent=group,
    )
    observations.add_data(
        {
            "Implicit scalar value": {
                "values": source_values,
            }
        }
    )

    folded_surface = Surface.create(
        workspace,
        name="RBF folded horizon - isovalue 0",
        vertices=folded_vertices,
        cells=folded_cells,
        parent=group,
    )
    folded_surface.add_data(
        {
            "RBF value on surface": {
                "values": fitted_values_at_vertices,
            },
            "True vertical error": {
                "values": true_vertical_errors,
            },
        }
    )

print("Created:", folded_output_path)

## What this baseline proves

This section demonstrates the complete technical path: scattered scalar observations -> ferreus_rbf scalar field -> ferreus_rmt zero-isosurface -> geoh5 objects for ANALYST.

It isn't yet a test of geological orientation or exclusion constraints. Every source point above was assigned a full scalar value from the known analytic field. Real geological inputs more commonly include:

- **contact/interface points**, which say f(p) = 0;
- **orientation measurements**, which say that the gradient of f should point along a measured normal; and
- **exclusion/inequality observations**, which specify only that a point must be on one side of the surface, such as f(p) > 0 or f(p) < 0.

The public ferreus_rbf Python constructor accepts point coordinates and scalar values; it does not expose separate derivative or inequality observations. The next POC sections can therefore test value-based encodings:

1. approximate an orientation at contact point p with offset samples p + delta times n and p - delta times n, assigned positive and negative values respectively; and
2. approximate an exclusion constraint with a signed pseudo-value on the required side of the surface.

Those are approximations that should be measured against this baseline.

## Compare the existing marching-cubes method with Ferreus RMT

This is an extraction-algorithm comparison. Both algorithms receive the same fitted RBF field, model bounds, lattice spacing, and isovalue. That keeps the scalar model fixed so differences in the resulting meshes come from marching cubes versus RMT rather than from different geological interpolation methods.

The existing application first samples data onto a complete regular grid and then calls scikit-image marching cubes. The code below performs that same dense-grid extraction. RMT instead evaluates the field while following the surface outward from seed points.

This comparison does not yet reproduce every part of the existing application's preprocessing, such as weighted averaging, maximum distance, or topographic masking. A later end-to-end test can feed both implementations the same real geoh5 object.

In [ ]:
from time import perf_counter

from skimage.measure import marching_cubes


# Construct the same kind of dense regular grid used by the existing
# iso_surfaces application. The array axes are X, Y, then Z.
comparison_grid = [
    np.arange(
        MODEL_EXTENTS[axis_index],
        MODEL_EXTENTS[axis_index + 3] + 0.5 * folded_resolution,
        folded_resolution,
        dtype=np.float64,
    )
    for axis_index in range(3)
]

comparison_x, comparison_y, comparison_z = np.meshgrid(
    comparison_grid[0],
    comparison_grid[1],
    comparison_grid[2],
)
comparison_grid_points = np.column_stack(
    [
        comparison_x.ravel(),
        comparison_y.ravel(),
        comparison_z.ravel(),
    ]
)

marching_cubes_total_start = perf_counter()

grid_evaluation_start = perf_counter()
comparison_grid_values = np.asarray(
    rbf_model.evaluate_targets(comparison_grid_points),
    dtype=np.float64,
).reshape(comparison_x.shape)
grid_evaluation_seconds = perf_counter() - grid_evaluation_start

marching_cubes_start = perf_counter()
mc_index_vertices, mc_cells, _mc_normals, _mc_values = marching_cubes(
    comparison_grid_values,
    level=folded_isovalue,
)

# marching_cubes returns positions in array-index coordinates. Map each
# axis back to the original X, Y, and Z coordinates, matching the logic
# in surface_apps.iso_surfaces.utils.extract_iso_surfaces.
mc_vertices = np.column_stack(
    [
        np.interp(
            mc_index_vertices[:, axis_index],
            np.arange(comparison_grid[axis_index].size),
            comparison_grid[axis_index],
        )
        for axis_index in range(3)
    ]
).astype(np.float64)
mc_cells = np.asarray(mc_cells, dtype=np.uint32)

marching_cubes_algorithm_seconds = perf_counter() - marching_cubes_start
marching_cubes_total_seconds = perf_counter() - marching_cubes_total_start

print("Regular-grid shape:", comparison_grid_values.shape)
print("Regular-grid sample count:", comparison_grid_points.shape[0])
print("Marching-cubes vertices:", mc_vertices.shape)
print("Marching-cubes triangles:", mc_cells.shape)
print("Grid evaluation seconds:", grid_evaluation_seconds)
print("Marching-cubes algorithm seconds:", marching_cubes_algorithm_seconds)
print("Marching-cubes total seconds:", marching_cubes_total_seconds)

### Time RMT against the same field

The RMT timing includes its field evaluations and mesh construction. The RBF evaluator is already cached for both methods. These single-run timings are useful for orientation, but they are not a rigorous performance benchmark; repeated warm runs and larger datasets would be needed for that.

In [ ]:
rmt_total_start = perf_counter()

comparison_rmt_mesh = build_isosurface(
    seed_points=folded_seed_points,
    extents=MODEL_EXTENTS,
    resolution=folded_resolution,
    isovalue=folded_isovalue,
    surface_fn=rbf_model.evaluate_targets,
    gradient_fn=rbf_model.evaluate_targets_with_gradients,
    cluster_method=ClusterMethod.CurvatureWeighted,
    boundary_closure=BoundaryClosure.None_,
)

rmt_total_seconds = perf_counter() - rmt_total_start

comparison_rmt_vertices = np.asarray(
    comparison_rmt_mesh.vertices,
    dtype=np.float64,
)
comparison_rmt_cells = np.asarray(
    comparison_rmt_mesh.facets,
    dtype=np.uint32,
)

print("RMT vertices:", comparison_rmt_vertices.shape)
print("RMT triangles:", comparison_rmt_cells.shape)
print("RMT total seconds:", rmt_total_seconds)

### Compare geometry and topology

The triangle-quality score below is 1.0 for an equilateral triangle and approaches zero for a very thin or degenerate triangle. The 5th percentile is particularly useful because it describes the poor-quality end of each mesh without depending on only one extreme triangle.

An edge used by more than two triangles is non-manifold. Boundary edges are expected here because the folded horizon reaches the sides of the extraction box and boundary closure was disabled. Therefore, a non-zero boundary-edge count does not by itself indicate a broken mesh.

In [ ]:
def calculate_mesh_statistics(
    mesh_vertices: np.ndarray,
    mesh_cells: np.ndarray,
    elapsed_seconds: float,
) -> tuple[dict[str, float | int], np.ndarray, np.ndarray]:
    """Calculate comparable geometric and topological mesh diagnostics."""
    triangles = mesh_vertices[mesh_cells]

    edge_01 = triangles[:, 1] - triangles[:, 0]
    edge_12 = triangles[:, 2] - triangles[:, 1]
    edge_20 = triangles[:, 0] - triangles[:, 2]

    squared_edge_sum = (
        np.sum(edge_01**2, axis=1)
        + np.sum(edge_12**2, axis=1)
        + np.sum(edge_20**2, axis=1)
    )
    twice_area = np.linalg.norm(
        np.cross(edge_01, -edge_20),
        axis=1,
    )
    triangle_quality = np.divide(
        2.0 * np.sqrt(3.0) * twice_area,
        squared_edge_sum,
        out=np.zeros_like(twice_area),
        where=squared_edge_sum > 0.0,
    )

    mesh_edges = np.vstack(
        [
            mesh_cells[:, [0, 1]],
            mesh_cells[:, [1, 2]],
            mesh_cells[:, [2, 0]],
        ]
    )
    mesh_edges.sort(axis=1)
    _unique_edges, edge_use_counts = np.unique(
        mesh_edges,
        axis=0,
        return_counts=True,
    )

    fitted_mesh_values = np.asarray(
        rbf_model.evaluate(mesh_vertices),
        dtype=np.float64,
    ).reshape(-1)
    mesh_true_vertical_errors = true_geological_field(mesh_vertices)

    statistics = {
        "vertices": int(mesh_vertices.shape[0]),
        "triangles": int(mesh_cells.shape[0]),
        "elapsed_seconds": float(elapsed_seconds),
        "quality_mean": float(np.mean(triangle_quality)),
        "quality_5th_percentile": float(np.percentile(triangle_quality, 5)),
        "boundary_edges": int(np.count_nonzero(edge_use_counts == 1)),
        "non_manifold_edges": int(np.count_nonzero(edge_use_counts > 2)),
        "mean_absolute_isovalue_error": float(
            np.mean(np.abs(fitted_mesh_values - folded_isovalue))
        ),
        "mean_absolute_vertical_error": float(
            np.mean(np.abs(mesh_true_vertical_errors))
        ),
        "maximum_absolute_vertical_error": float(
            np.max(np.abs(mesh_true_vertical_errors))
        ),
    }

    return statistics, fitted_mesh_values, mesh_true_vertical_errors


mc_statistics, mc_fitted_values, mc_true_vertical_errors = calculate_mesh_statistics(
    mc_vertices,
    mc_cells,
    marching_cubes_total_seconds,
)
rmt_statistics, rmt_fitted_values, rmt_true_vertical_errors = calculate_mesh_statistics(
    comparison_rmt_vertices,
    comparison_rmt_cells,
    rmt_total_seconds,
)

comparison_rows = [
    ("Vertices", "vertices", ".0f"),
    ("Triangles", "triangles", ".0f"),
    ("Total time (seconds)", "elapsed_seconds", ".4f"),
    ("Mean triangle quality", "quality_mean", ".4f"),
    ("5th-percentile quality", "quality_5th_percentile", ".4f"),
    ("Boundary edges", "boundary_edges", ".0f"),
    ("Non-manifold edges", "non_manifold_edges", ".0f"),
    ("Mean absolute isovalue error", "mean_absolute_isovalue_error", ".4f"),
    ("Mean absolute vertical error", "mean_absolute_vertical_error", ".4f"),
    ("Maximum absolute vertical error", "maximum_absolute_vertical_error", ".4f"),
]

print(f"{'Metric':38s} {'Marching cubes':>18s} {'Ferreus RMT':>18s}")
print("-" * 76)
for label, key, value_format in comparison_rows:
    mc_text = format(mc_statistics[key], value_format)
    rmt_text = format(rmt_statistics[key], value_format)
    print(f"{label:38s} {mc_text:>18s} {rmt_text:>18s}")

### View the two extraction results

The surfaces should have the same large-scale folded geometry because they represent the same zero-isovalue of the same RBF. Differences to look for are triangulation density, skinny triangles, boundary shape, and small positional deviations introduced by RMT's regularisation and vertex clustering, though those things are difficult to discern when there are so many triangles.

In [ ]:
figure = plt.figure(figsize=(14, 6))

comparison_meshes = [
    ("Existing method: marching cubes", mc_vertices, mc_cells, "darkorange"),
    ("Ferreus RMT", comparison_rmt_vertices, comparison_rmt_cells, "steelblue"),
]

for plot_index, (title, plot_vertices, plot_cells, colour) in enumerate(
    comparison_meshes,
    start=1,
):
    axis = figure.add_subplot(1, 2, plot_index, projection="3d")
    axis.plot_trisurf(
        plot_vertices[:, 0],
        plot_vertices[:, 1],
        plot_vertices[:, 2],
        triangles=plot_cells,
        color=colour,
        linewidth=0.05,
        alpha=0.85,
    )
    axis.scatter(
        folded_seed_points[:, 0],
        folded_seed_points[:, 1],
        folded_seed_points[:, 2],
        color="black",
        s=8,
        label="RMT seeds",
    )
    axis.set_xlim(MODEL_EXTENTS[[0, 3]])
    axis.set_ylim(MODEL_EXTENTS[[1, 4]])
    axis.set_zlim(MODEL_EXTENTS[[2, 5]])
    axis.set_box_aspect(MODEL_EXTENTS[3:] - MODEL_EXTENTS[:3])
    axis.set_xlabel("X")
    axis.set_ylabel("Y")
    axis.set_zlabel("Z")
    axis.view_init(elev=25, azim=-55)
    axis.set_title(title)

figure.tight_layout()
plt.show()

### Export both comparison surfaces to geoh5

Both meshes are placed in the same group so they can be toggled, coloured, and compared in Geoscience ANALYST. Each receives the RBF value and known vertical error at its vertices.

In [ ]:
comparison_output_path = output_directory / (
    f"marching_cubes_vs_rmt_{datetime.now():%Y%m%d_%H%M%S}.geoh5"
)

with Workspace.create(comparison_output_path) as workspace:
    comparison_group = ContainerGroup.create(
        workspace,
        name="Marching cubes versus Ferreus RMT",
    )

    mc_surface = Surface.create(
        workspace,
        name="Existing method - marching cubes",
        vertices=mc_vertices,
        cells=mc_cells,
        parent=comparison_group,
    )
    mc_surface.add_data(
        {
            "RBF value on surface": {"values": mc_fitted_values},
            "True vertical error": {"values": mc_true_vertical_errors},
        }
    )

    rmt_surface = Surface.create(
        workspace,
        name="Ferreus RMT",
        vertices=comparison_rmt_vertices,
        cells=comparison_rmt_cells,
        parent=comparison_group,
    )
    rmt_surface.add_data(
        {
            "RBF value on surface": {"values": rmt_fitted_values},
            "True vertical error": {"values": rmt_true_vertical_errors},
        }
    )

print("Created:", comparison_output_path)

### What this comparison does and does not establish

This establishes whether RMT is a credible alternative mesh extractor for a field that already exists. It can compare mesh density, triangle quality, topology, approximate accuracy, and timing under controlled conditions.

It does not establish support for geological orientation or exclusion constraints. Those affect how the scalar field is fitted from observations, which happens before either marching cubes or RMT is called.